# BER: Business Entity Resolution on Kaggle (GPU)

Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (to clone the code), and
**Persistence = Files** if you want `models/` to survive a restart. Add the competition data as a
Kaggle dataset (the `student_resource` folder). The paths are found automatically below.

Budget: training takes ~2.5-3.5 h (blocking + features ~1.5 h, GBMs on GPU, cross-encoder ~30-45 min),
and inference ~4-4.5 h. Run training and inference in one session, or save `models/` in between.

In [ ]:
!nvidia-smi -L
!free -g | head -2; nproc; df -h /kaggle/working | tail -1

## 1. Code

In [ ]:
import os
REPO = "https://github.com/MounishSenisetty/BER.git"   # private repo: https://<user>:<token>@github.com/...
BRANCH = "claude/cool-newton-qy2lr2"
CODE = "/kaggle/working/BER"
if not os.path.exists(CODE):
    # GIT_LFS_SKIP_SMUDGE: do not download the 2.4 GB dataset stored in the repo; the Kaggle dataset is used
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b $BRANCH $REPO $CODE
!cd $CODE && git log --oneline -1

In [ ]:
# torch, xgboost (CUDA), lightgbm, scikit-learn, pandas are preinstalled on the Kaggle GPU image
!pip install -q rapidfuzz jellyfish faiss-cpu 2>&1 | tail -1
import torch, xgboost
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), "GPU(s)")
print("xgboost", xgboost.__version__, "CUDA build:", xgboost.build_info().get("USE_CUDA"))

## 2. Data paths

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "add the competition dataset (student_resource) to the notebook"
DATA = os.path.dirname(os.path.dirname(hits[0]))          # .../student_resource/dataset
TRAIN, TEST = f"{DATA}/train", f"{DATA}/test"
ROOT = os.path.dirname(DATA)                              # .../student_resource (has utils/)
WORK = "/kaggle/working"
os.environ.update(TRAIN=TRAIN, TEST=TEST, ROOT=ROOT, WORK=WORK, CODE=CODE)
print(TRAIN, TEST, sep="\n"); print(os.listdir(TRAIN), os.listdir(TEST))

## 3. Train

* S1 dropout is sized automatically from `../test` (test has more distractor records per entity).
* The cross-encoder turns on automatically with a CUDA GPU. Training keeps the best of
  stage1 / stage2 / stage2+ce by nested-CV macro F0.5.
* The log ends with the **loss report** (points lost per error type). Please share it.

In [ ]:
%%time
!cd $CODE && python -u -m src.train --data-dir "$TRAIN" --model-dir $WORK/models --cache-dir /tmp/ber_s2 \
    2>&1 | tee $WORK/train.log | grep -vE "prepare \+ block \.\.\.$|features for .* \.\.\.$|  CE\[.\] step"

In [ ]:
import json
r = json.load(open(f"{WORK}/models/cv_report.json"))
print("blocking:", {k: v for k, v in r["blocking"].items() if k != "recall_at_cap"})
print("variants (nested-CV macro F0.5):", r.get("variants"), "-> final:", r.get("final"))
print("S1 dropout:", r.get("s1_drop"), "| lexicon entries:", r.get("lexicon_size"))
!sed -n '/Loss report/,/Examples FP/p' $WORK/train.log | head -40

## 4. Inference on the test split

In [ ]:
%%time
!cd $CODE && python -u -m src.inference --data-dir "$TEST" --model-dir $WORK/models --out-dir $WORK/output \
    2>&1 | tee $WORK/inference.log | grep -vE "prepare \+ block \.\.\.$|scoring for .* \.\.\.$"

## 5. Validate and package

In [ ]:
!python "$ROOT/utils/validate_submission.py" --matching $WORK/output/matching_results.tsv \
      --candidate $WORK/output/candidate_pairs.tsv --test-dir "$TEST"

In [ ]:
!cd $CODE && python scripts/make_submission_zip.py --team YOUR_TEAM --out-dir $WORK/output \
      --doc Documentation_template.md --dest $WORK
!ls -la $WORK/*.zip